# CASMI26 Engine 2 の自前 2 表の生成（#74）
生成コードは `taichiiiii/casmi26-engine2-build-code`（リポジトリの `scripts/engine2/`、#72）。v4g の中の元の 2 表は読まない（明示パス、ファイル名ガード、読んだファイルの一覧）。

In [ ]:
N_MOL = 50
REPEAT = 1


In [ ]:
# 入力の場所（明示パス）と RDKit 2026.03.3
import os, sys, json, glob, subprocess, time, resource
from pathlib import Path
T0 = time.time()
IN = Path('/kaggle/input')
def one(pattern, must=None):
    hits = sorted(p for p in IN.rglob(pattern) if must is None or must in str(p))
    assert len(hits) >= 1, (pattern, must)
    return min(hits, key=lambda p: len(str(p)))
V4G = one('casmi26-v3-models/code/casmi/chem.py', 'casmi26-attributed-v4g-reproduction-assets').parents[3]
FS = one('fpnet_fold1.pt', 'casmi26-fold-safe-fpnet').parent
FOLDS = one('structures.parquet', 'casmi26-identity-disjoint-validation-folds').parent
CODE = one('build_tables.py', 'casmi26-engine2-build-code').parent
COMP = min((p for p in IN.rglob('train.parquet') if 'molecule-id-mass-spectra' in str(p).lower()), key=lambda p: len(str(p))).parent
import shutil
shutil.copytree(CODE, '/kaggle/working/e2code', dirs_exist_ok=True)   # numba の cache を書ける場所で動かす
CODE = Path('/kaggle/working/e2code')
TABLES_PRESENT = sorted(str(p) for p in IN.rglob('*') if p.is_file() and (p.name == 'rank_train.npz' or p.name.startswith('sim_rank_rows')))
print('original Engine 2 tables under /kaggle/input (v4g の写し、読まない):', TABLES_PRESENT)
assert all('casmi26-attributed-v4g-reproduction-assets' in p for p in TABLES_PRESENT), TABLES_PRESENT
from importlib.metadata import version as _v
try:
    installed = _v('rdkit')
except Exception:
    installed = ''
if installed.replace('.03.', '.3.') != '2026.3.3':
    tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
    wheels = sorted(IN.rglob(f'rdkit-2026.3.3-{tag}-*.whl'))
    assert wheels, f'RDKit 2026.3.3 wheel for {tag}'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '-q', str(wheels[0])])
ASSETS = dict(coco_dir=str(V4G / 'coconut-casmi26-candidates'), bio_dir=str(V4G / 'casmi-chebi-lmsd-structures'),
              pc_dir=str(V4G / 'casmi26-pubchemlite-cc0-candidate-fingerprints'), fs_models_dir=str(FS),
              fs_bits=str(FS / 'fp_bits.npy'), casmi_code=str(V4G / 'casmi26-v3-models' / 'code'),
              split=str(FS / 'split.parquet'), structures=str(FOLDS / 'structures.parquet'),
              drop_keys_dir=str(CODE.parent / 'drop_keys') if (CODE.parent / 'drop_keys').is_dir() else str(CODE / 'drop_keys'))
for k, v in ASSETS.items():
    assert os.path.exists(v), (k, v)
json.dump(ASSETS, open('/kaggle/working/assets.json', 'w'), indent=1)
print(json.dumps(ASSETS, indent=1))


In [ ]:
# 生成（サブプロセス。PYTHONHASHSEED=0、ログは build.log）
OUT = '/kaggle/working/own'
cmd = [sys.executable, str(CODE / 'build_tables.py'), '--train', str(COMP / 'train.parquet'), '--assets', '/kaggle/working/assets.json',
       '--out', OUT, '--n-mol', str(N_MOL), '--repeat', str(REPEAT), '--ab-check']
print(cmd, flush=True)
with open('/kaggle/working/build.log', 'w') as lf:
    rc = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT, cwd=str(CODE),
                        env={**os.environ, 'PYTHONHASHSEED': '0', 'OPENBLAS_NUM_THREADS': '1'}).returncode
PEAK_GB = resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6   # Linux は KB
log = open('/kaggle/working/build.log').read()
print(log[-6000:])
print('child peak RSS GB', round(PEAK_GB, 2), 'rc', rc)
assert rc == 0, rc


In [ ]:
# 確認（形・正例・恒等式・決定性・読んだファイル）
import numpy as np, importlib.util
spec = importlib.util.spec_from_file_location('sch', str(CODE / 'engine2_table_schema.py'))
SCH = importlib.util.module_from_spec(spec); spec.loader.exec_module(SCH)
rep = json.load(open(f'{OUT}/build_report.json'))
chk = dict(n_mol=N_MOL, repeat=REPEAT, child_peak_rss_gb=round(PEAK_GB, 2), opened_forbidden=rep['opened_forbidden'], deterministic=rep.get('deterministic'),
           runs=[], wall_secs=round(time.time() - T0, 1))
for r in rep['runs']:
    for path, blocks in ((r['files'][0], None), (r['files'][1], ['base'])):
        d = SCH.describe(path, blocks)
        fr = {k: v.get('frac', v.get('status')) for k, v in d['identities'].items()}
        chk['runs'].append(dict(file=os.path.basename(path), n_rows=d['n_rows'], n_feat=d['n_feat'], blocks=d['blocks'],
                                pos_per_group=d['pos_per_group'], n_groups=d['n_groups'],
                                identities_all_one=all(v in (1.0, 'all_zero') for v in fr.values()), identities=fr))
json.dump(chk, open('/kaggle/working/check.json', 'w'), indent=1)
print(json.dumps(chk, indent=1)[:6000])
print('report:', json.dumps({k: rep[k] for k in rep if k not in ('opened_files', 'runs')}, indent=1)[:4000])
for r in rep['runs']:
    print('run', {k: r[k] for k in r if k != 'queries'})


In [ ]:
# 高速化の前後の比較（(i) channels_hidden 2 回 対 (ii) channels_pair、z は共有）と z の 2 回、CPU 情報
ab = rep['ab_check']
chk['ab_check'] = ab
chk['cpu_info'] = {k: rep['cpu_info'][k] for k in rep['cpu_info'] if k != 'torch_config'}
json.dump(chk, open('/kaggle/working/check.json', 'w'), indent=1)
print(json.dumps(ab, indent=1))
print(rep['cpu_info']['lscpu'])
print(rep['cpu_info']['torch_config'])
print('torch threads', rep['cpu_info']['torch_num_threads'], 'os.cpu_count', rep['cpu_info']['os_cpu_count'])
